# 6. Robustness: technology aggregation and long differences

Notebook 5 produced a large cross-sectional effect (`net_any` 0.061\*\*\*) that disappears
under firm-subclass FE. The collapse is consistent with sorting, but the within-pair arm is
too thin to say more: 390 moving pairs, 163 with the outcome also moving, and an interval
that excludes the cross-sectional magnitude without excluding a moderate effect ("cross-sectional" here means the contrast is across technologies within a firm-year, not
that the data are a cross-section — the panel is the same in both specifications).

Both variants here attack that thinness from different directions.

**Coarser technology class.** IPC 3-character class instead of 4-character subclass. More
firms and more co-affiliates share each technology, so both arms gain variation — and
crucially, more firm-subclass pairs should see the network stock move within the window.
The cost is interpretive: a 3-character class is a broad field (C07 is organic chemistry,
H04 is electrical communication), so "the network patents in the same technology" becomes a
weaker statement about knowledge proximity.

**Long differences.** Collapse 2015–2024 into two periods and compare. Ten annual
observations of a near-monotone cumulative stock carry heavy serial correlation; a single
pre/post contrast trades them for one cleaner comparison. Standard in this literature when
the annual panel is noisy.

Neither is a reinterpretation of notebook 5. Both ask whether the time-dimension null
survives a specification with more power to detect an effect.

**Build strategy.** The 3-character panel is a collapse of the existing one, not a rebuild.
Notebook 3 assigned each patent family a single modal IPC subclass, so every family sits in
exactly one 4-character subclass and therefore exactly one 3-character class. `net_stock`
counts distinct families, so summing across the 4-char subclasses within a class double-counts
nothing; `patented` becomes a max. The next cell verifies that assumption rather than
assuming it.

In [1]:
import numpy as np
import pandas as pd
import pyfixest as pf

data_dir = r"C:\Users\Popov\Documents\Studies\IMT_studies\Projects\Econ_2\Datasets"
panel = pd.read_parquet(data_dir + r"\panel_firm_subclass_year.parquet")

FIRM, PARENT, SUB, YEAR = "bvd_key", "parent_key", "subclass", "year"
Y, X = "patented", "net_stock"

panel["cls"] = panel[SUB].str[:3]          # IPC class, e.g. C07D -> C07

p3 = (panel.groupby([FIRM, PARENT, "cls", YEAR, "parent_is_foreign"], as_index=False)
            .agg(patented=(Y, "max"), net_stock=(X, "sum"), n_new=("n_new", "sum")))

p3["net_any"] = (p3[X] > 0).astype("int8")
p3["log_net"] = np.log1p(p3[X])
p3["firm_year"] = p3[FIRM] + "|" + p3[YEAR].astype(str)
p3["cls_year"]  = p3["cls"] + "|" + p3[YEAR].astype(str)
p3["pair"]      = p3[FIRM] + "|" + p3["cls"]
p3["firm_id"]   = p3[FIRM].astype("category").cat.codes

print(f"4-char: {len(panel):,} cells | {panel[SUB].nunique():,} subclasses")
print(f"3-char: {len(p3):,} cells | {p3['cls'].nunique():,} classes | {p3[FIRM].nunique():,} firms")
print(f"outcome mean {p3[Y].mean():.4f} | exposed cells {(p3[X] > 0).mean():.3f}")

4-char: 94,010 cells | 527 subclasses
3-char: 58,600 cells | 123 classes | 674 firms
outcome mean 0.0463 | exposed cells 0.616


In [2]:
# If each family has one subclass, summing net_stock within a class cannot exceed the
# number of distinct network families the firm has in that class. The sharpest available
# check: the collapse must preserve total new families and must not lose any patenting.
print("n_new totals match:", panel["n_new"].sum() == p3["n_new"].sum())
print("patenting cells, 4-char:", f"{panel[Y].sum():,}",
      "| 3-char:", f"{p3[Y].sum():,}", "(3-char <= 4-char expected)")

# Cells per class-year should be the number of firms active there; spot-check the grid
print("\nmax cells per firm-class-year (must be 1):",
      p3.groupby([FIRM, "cls", YEAR]).size().max())
print("years per pair:", p3.groupby("pair")[YEAR].nunique().value_counts().to_dict())

n_new totals match: True
patenting cells, 4-char: 3,406 | 3-char: 2,715 (3-char <= 4-char expected)

max cells per firm-class-year (must be 1): 1
years per pair: {10: 5860}


The collapse is clean and substantial. 527 subclasses become 123 classes, 94,010 cells
become 58,600, and the outcome mean rises from 3.6% to 4.6% — a firm patenting in two
subclasses of the same class now registers as one active cell rather than two, so each
remaining cell is more likely to be a 1. Exposure is near-unchanged at 61.6%.

The checks pass: `n_new` totals match exactly, so no family is lost or duplicated; patenting
cells fall from 3,406 to 2,715, the expected direction; one cell per firm-class-year
confirms the keys are complete; and all 5,860 pairs have the full ten years, so the grid is
balanced.

Whether this buys the within-pair variation notebook 5 lacked is the next check.

### Risk set at 3-character class

Same rule as notebook 5 — keep firm × class combinations the firm patented in before 2015 —
rebuilt at the coarser level. A firm with pre-2015 history in any subclass of C07 now has
the whole class in its risk set, so the risk set should be wider than notebook 5's 3,311
pairs even though there are fewer pairs overall.

In [3]:
patents = pd.read_stata(data_dir + r"\patents.dta")
PFIRM, PFAM, PDATE = "bvd_key", "family_identifier__simple", "priority_date"

valid = patents.dropna(subset=[PDATE, PFAM])
pre = valid[valid[PDATE].dt.year < 2015].copy()
pre["cls"] = pre["ipc_code__main"].str[:3]

rs3 = (pre[pre[PFIRM].isin(set(p3[FIRM]))][[PFIRM, "cls"]]
         .drop_duplicates())
rs3 = rs3[(rs3["cls"] != "") & rs3["cls"].isin(p3["cls"].unique())]

print(f"risk-set pairs: {len(rs3):,} | firms: {rs3[PFIRM].nunique():,} | "
      f"classes: {rs3['cls'].nunique():,}")

rs_pairs3 = set(zip(rs3[PFIRM], rs3["cls"]))
rsp3 = p3[[(f, c) in rs_pairs3 for f, c in zip(p3[FIRM], p3["cls"])]].copy()

fy = rsp3.groupby([FIRM, YEAR])["cls"].nunique()
print(f"\nrisk-set panel: {len(rsp3):,} cells | {rsp3[FIRM].nunique():,} firms | "
      f"{rsp3['pair'].nunique():,} pairs")
print(f"firm-years with >=2 classes: {(fy >= 2).sum():,} of {len(fy):,} ({(fy >= 2).mean():.1%})")
print(f"risk-set cells with net_stock>0: {(rsp3[X] > 0).mean():.1%}")

risk-set pairs: 2,109 | firms: 516 | classes: 120

risk-set panel: 18,570 cells | 516 firms | 1,857 pairs
firm-years with >=2 classes: 3,070 of 5,160 (59.5%)
risk-set cells with net_stock>0: 33.2%


Coarsening shrinks the risk set rather than widening it: 1,857 pairs against notebook 5's
3,311, on 516 firms either way. Collapsing subclasses into classes merges a firm's several
pre-2015 technologies into one, so each firm brings fewer distinct cells. Firm-years with
≥2 classes fall to 59.5% from 64.1%, and exposure rises to 33.2% from 24.5% — a broader
class is more likely to contain a sibling's patent.

Fewer pairs but more exposure variation per pair. Whether that nets out in favour of the
within-pair arm is what the next cell decides.

### Identifying variation — the decisive check

This is why the notebook exists. Notebook 5's pair-FE arm rested on 390 moving pairs (163
with the outcome also moving) and 127 (56) for the binary form. If coarsening does not
raise those counts materially, the time-dimension question stays bounded and the long
differences are the only remaining lever.

In [4]:
for v in [X, "log_net", "net_any"]:
    fy = rsp3.groupby(["firm_id", YEAR]).agg(x=(v, "nunique"), y=(Y, "nunique"))
    pr = rsp3.groupby("pair").agg(x=(v, "nunique"), y=(Y, "nunique"))
    print(f"{v:10s} | firm-years: x varies {(fy.x > 1).sum():5,} of {len(fy):,}, "
          f"y also {((fy.x > 1) & (fy.y > 1)).sum():5,} "
          f"|| pairs: x moves {(pr.x > 1).sum():5,} of {len(pr):,}, "
          f"y also {((pr.x > 1) & (pr.y > 1)).sum():4,}")

net_stock  | firm-years: x varies 2,206 of 5,160, y also   655 || pairs: x moves   381 of 1,857, y also  169
log_net    | firm-years: x varies 2,206 of 5,160, y also   655 || pairs: x moves   381 of 1,857, y also  169
net_any    | firm-years: x varies 1,845 of 5,160, y also   594 || pairs: x moves   107 of 1,857, y also   49


Sad. Coarsening does not buy the within-pair variation. 381 moving pairs against notebook 5's
390, and 169 with the outcome also moving against 163 — the same counts. The *share* of
pairs that move rises (20.5% from 14.2%) only because the denominator shrank.

The implication is structural rather than specificational: roughly 380 firm-technology pairs
see their network stock change within 2015–2024, and aggregating technologies repackages
those changes rather than creating new ones. Long differences would reduce noise inside
those pairs, not add pairs, so they cannot fix it either.

The time-dimension question therefore stays bounded as notebook 5 left it. What this
notebook can still do is test whether the cross-sectional result — the one that is actually
estimated precisely — survives coarsening.

In [5]:
fe1 = "firm_year + cls_year"
fe2 = "firm_year + cls_year + pair"
cl = {"CRV1": "firm_id"}

a3, b3 = [], []
for v in [X, "log_net", "net_any"]:
    a3.append(pf.feols(f"{Y} ~ {v} | {fe1}", data=rsp3, vcov=cl))
    b3.append(pf.feols(f"{Y} ~ {v} | {fe2}", data=rsp3, vcov=cl))

pf.etable(a3 + b3)

c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\

GT(_tbl_data=  __index_level_0__ __index_level_1__                     0  \
0              coef         net_stock  0.000** <br> (0.000)   
1              coef           log_net                         
2              coef           net_any                         
3                fe          cls_year                     -   
4                fe         firm_year                     -   
5                fe              pair                     -   
6             stats      Observations                16,390   
7             stats                R²                  0.31   

                       1                      2                    3  \
0                                                -0.000 <br> (0.000)   
1  0.057*** <br> (0.009)                                               
2                         0.107*** <br> (0.017)                        
3                      -                      -                    -   
4                      -                      -                    -   
5                      -                      -                    -   
6                 16,390                 16,390               16,390   
7                  0.329                  0.322                0.623   

                     4                    5  
0                                            
1  -0.014 <br> (0.021)                       
2                       -0.019 <br> (0.023)  
3                    -                    -  
4                    -                    -  
5                    -                    -  
6               16,390               16,390  
7                0.623                0.623  , _body=<great_tables._gt_data.Body object at 0x000001C98E967290>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(4)'), column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(5)'), column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(6)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001C98E956A80>, _spanners=Spanners([SpannerInfo(spanner_id='patented', spanner_level=1, spanner_label=Html(text='patented'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2', '3', '4', '5'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001C98E956540>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001C98E9561B0>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001C98E955EB0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001C98E956390>, <great_tables._gt_data.FormatInfo object at 0x000001C98E956630>, <great_tables._gt_data.FormatInfo object at 0x000001C98E956600>, <great_tables._gt_data.FormatInfo object at 0x000001C98E9544D0>, <great_tables._gt_data.FormatInfo object at 0x000001C98E954980>, <great_tables._gt_data.FormatInfo object at 0x000001C98E954950>, <great_tables._gt_data.FormatInfo object at 0x000001C98E9541D0>

In [6]:
def cls_table(models, pair_fe):
    t = pf.etable(models, type="df").drop("fe", level=0)
    t.loc[("fe", "Firm x year FE"), :]  = ["Yes"] * 3
    t.loc[("fe", "Class x year FE"), :] = ["Yes"] * 3
    t.loc[("fe", "Firm-class FE"), :]   = [pair_fe] * 3
    return (t.reindex(["coef", "fe", "stats"], level=0)
             .replace({r"\n": " "}, regex=True))

t3a = cls_table(a3, "No")
t3b = cls_table(b3, "Yes")
t3a

patented                      \
                                     (1)                 (2)   
coef  net_stock        0.000**   (0.000)                       
      log_net                             0.057***   (0.009)   
      net_any                                                  
fe    Firm x year FE                 Yes                 Yes   
      Class x year FE                Yes                 Yes   
      Firm-class FE                   No                  No   
stats Observations                16,390              16,390   
      R²                            0.31               0.329   

                                           
                                      (3)  
coef  net_stock                            
      log_net                              
      net_any          0.107***   (0.017)  
fe    Firm x year FE                  Yes  
      Class x year FE                 Yes  
      Firm-class FE                    No  
stats Observations                 16,390  
      R²                            0.322

In [7]:
for t, lab, cap in [(t3a, "tab:cls_a", "Firm x year + class x year FE"),
                    (t3b, "tab:cls_b", "Adding firm-class FE")]:
    o = t.copy()
    o.index = o.index.droplevel(0)
    o = o.rename(index={X: "Network stock", "log_net": "log(1 + stock)",
                        "net_any": "Network present", "R²": "$R^2$"})
    print(o.to_latex(escape=False, column_format="l" + "c" * 3,
                     caption=f"IPC 3-character class — {cap}", label=lab))
    print()

\begin{table}
\caption{IPC 3-character class — Firm x year + class x year FE}
\label{tab:cls_a}
\begin{tabular}{lccc}
\toprule
 & \multicolumn{3}{r}{patented} \\
 & (1) & (2) & (3) \\
\midrule
Network stock & 0.000**   (0.000) &  &  \\
log(1 + stock) &  & 0.057***   (0.009) &  \\
Network present &  &  & 0.107***   (0.017) \\
Firm x year FE & Yes & Yes & Yes \\
Class x year FE & Yes & Yes & Yes \\
Firm-class FE & No & No & No \\
Observations & 16,390 & 16,390 & 16,390 \\
$R^2$ & 0.31 & 0.329 & 0.322 \\
\bottomrule
\end{tabular}
\end{table}


\begin{table}
\caption{IPC 3-character class — Adding firm-class FE}
\label{tab:cls_b}
\begin{tabular}{lccc}
\toprule
 & \multicolumn{3}{r}{patented} \\
 & (1) & (2) & (3) \\
\midrule
Network stock & -0.000   (0.000) &  &  \\
log(1 + stock) &  & -0.014   (0.021) &  \\
Network present &  &  & -0.019   (0.023) \\
Firm x year FE & Yes & Yes & Yes \\
Class x year FE & Yes & Yes & Yes \\
Firm-class FE & Yes & Yes & Yes \\
Observations & 16,390 & 16,390 &

### What the 3-character tables show

The cross-sectional effect does not merely survive coarsening — it roughly doubles.
`net_any` goes from 0.061\*\*\* (0.015) at subclass level to 0.107\*\*\* (0.017) at class
level, and `log_net` from 0.025\*\* to 0.057\*\*\*. Against outcome means of 3.6% and 4.6%,
that is a rise from 1.7× to 2.3× the baseline rate in relative terms too.

That direction is informative, and it cuts against a knowledge-flow reading. Spillovers
should be *technologically local*: knowledge moves between closely related technologies, so
broadening the definition of "same technology" ought to dilute the measured effect by mixing
genuine proximity with mere co-membership of a wide field. Sorting predicts the opposite —
a firm and its siblings share a broad field far more reliably than a narrow one, so the
correlation strengthens as the class widens. The data behave as sorting predicts.

The within-pair arm is unchanged: −0.019 (0.023) and −0.014 (0.021), standard errors larger
than the point estimates, resting on the same ~380 moving pairs. Coarsening moved the
cross-sectional coefficient and left the time margin exactly where notebook 5 left it.

The levels form is again significant cross-sectionally (0.000\*\*) and again uninterpretable
at three decimals — treated as mechanical throughout, as in notebooks 4 and 5.

N is 16,390 after `pyfixest` drops 2,180 singleton fixed effects.

### The aggregation ladder

The jump from 0.061 at subclass level to 0.107 at class level is a direction, not yet a
result. Running the same specification at several levels of technological aggregation turns
it into one.

The two stories make opposite predictions:

- **Knowledge flow** is technologically local. Widening the definition of "same technology"
  mixes genuine proximity with mere co-membership of a broad field, so the measured effect
  should *dilute* as the class widens.
- **Sorting** is about shared identity. A firm and its siblings share a broad field far more
  reliably than a narrow one, so the correlation should *strengthen* with aggregation.

A monotone rise across IPC section (1 char, 8 fields), class (3 chars, 123) and subclass
(4 chars, 527) is evidence for the second. Note the comparison is of coefficients against
different outcome means, so the table reports both the coefficient and the effect relative
to the baseline rate.

This is a cross-sectional test — `firm_year + <level>_year` FE throughout — but the
pair-FE column is reported alongside so the contrast is visible at every level rather than
asserted. The moving-pair counts on the right show why that column stays flat: aggregation
repackages existing changes in network stock rather than creating new ones, so the number
of firm-technology pairs that identify the within-pair estimate barely moves.

In [8]:
def fmt(t):
    p = t["Pr(>|t|)"]
    s = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""
    return f"{t['Estimate']:.3f}{s} ({t['Std. Error']:.3f})"

LEVELS = {1: "section", 3: "class", 4: "subclass"}
rows = []

for k, name in LEVELS.items():
    d = panel.copy()
    d["lvl"] = d[SUB].str[:k]

    g = (d.groupby([FIRM, "lvl", YEAR], as_index=False)
           .agg(patented=(Y, "max"), net_stock=(X, "sum")))
    g["net_any"]   = (g[X] > 0).astype("int8")
    g["firm_year"] = g[FIRM] + "|" + g[YEAR].astype(str)
    g["lvl_year"]  = g["lvl"] + "|" + g[YEAR].astype(str)
    g["pair"]      = g[FIRM] + "|" + g["lvl"]
    g["firm_id"]   = g[FIRM].astype("category").cat.codes

    pre_k = pre.copy()
    pre_k["lvl"] = pre_k["ipc_code__main"].str[:k]
    keep = set(map(tuple, pre_k.loc[(pre_k["lvl"] != "") & pre_k[PFIRM].isin(set(g[FIRM])),
                                    [PFIRM, "lvl"]].drop_duplicates().values))
    g = g[[(f, l) in keep for f, l in zip(g[FIRM], g["lvl"])]].copy()

    m1 = pf.feols("patented ~ net_any | firm_year + lvl_year",
                  data=g, vcov={"CRV1": "firm_id"})
    m2 = pf.feols("patented ~ net_any | firm_year + lvl_year + pair",
                  data=g, vcov={"CRV1": "firm_id"})
    t1, t2 = m1.tidy().iloc[0], m2.tidy().iloc[0]

    pr = g.groupby("pair").agg(x=("net_any", "nunique"), y=("patented", "nunique"))
    ybar = g["patented"].mean()

    rows.append({"Level": f"{name} ({k})", "Units": g["lvl"].nunique(),
                 "Cells": f"{len(g):,}", "Mean $y$": f"{ybar:.3f}",
                 "No pair FE": fmt(t1), "Relative": f"{t1['Estimate'] / ybar:.2f}",
                 "With pair FE": fmt(t2),
                 "Moving pairs": int((pr.x > 1).sum()),
                 "$\\ldots y$ also": int(((pr.x > 1) & (pr.y > 1)).sum())})

ladder = pd.DataFrame(rows)
ladder

c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2520 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2520 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\

,Level,Units,Cells,Mean $y$,No pair FE,Relative,With pair FE,Moving pairs,$\ldots y$ also
0,section (1),8,"10,500",0.146,0.160*** (0.025),1.10,-0.011 (0.035),105,49
1,class (3),119,"18,570",0.098,0.107*** (0.017),1.10,-0.019 (0.023),107,49
2,subclass (4),492,"27,440",0.079,0.061*** (0.015),0.78,-0.014 (0.028),127,56


Read `Relative`, not the raw coefficient: outcome means differ across levels, so the
coefficients are not directly comparable. A rising relative effect with aggregation is the
sorting signature; a falling one would favour localized knowledge flow.

### The aggregation ladder

The raw coefficient rises monotonically with aggregation: 0.061 at subclass, 0.107 at
class, 0.160 at section. That is the sorting direction — a firm and its siblings share a
broad field more reliably than a narrow one, so the correlation strengthens as the category
widens. Localized knowledge flow predicts the opposite.

The relative effect is less clean. Outcome means rise with aggregation too (7.9%, 9.8%,
14.6%), and dividing through gives 0.78, 1.10, 1.10 — a step from subclass to class, then a
plateau. So the pattern is directionally consistent with sorting but it is not a smooth
gradient, and which normalization is the right comparison is arguable: the LPM coefficient
is a percentage-point effect, and both it and the base rate grow with aggregation. The
honest statement is that most of the movement happens between subclass and class, and
nothing reverses.

The pair-FE column is flat at every level: −0.014, −0.019, −0.011, standard errors roughly
twice the point estimates throughout. The moving-pair counts explain it — 127, 107, 105
pairs, with 56, 49, 49 where the outcome also moves. Aggregation changes the cross-sectional
estimate substantially and leaves the within-pair estimate untouched, because it repackages
existing changes in network stock rather than creating new ones. The time margin is
structurally thin in this data, not thin because of how technology was defined.

In [9]:
print(ladder.to_latex(index=False, escape=False, column_format="lrrrcccrr",
                      caption="Cross-sectional effect by level of technological aggregation",
                      label="tab:ladder"))
print(r"\footnotesize Dependent variable: new patent family in the cell. "
      r"Regressor: network present (0/1). SEs clustered by firm. "
      r"$^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$.")

\begin{table}
\caption{Cross-sectional effect by level of technological aggregation}
\label{tab:ladder}
\begin{tabular}{lrrrcccrr}
\toprule
Level & Units & Cells & Mean $y$ & No pair FE & Relative & With pair FE & Moving pairs & $\ldots y$ also \\
\midrule
section (1) & 8 & 10,500 & 0.146 & 0.160*** (0.025) & 1.10 & -0.011 (0.035) & 105 & 49 \\
class (3) & 119 & 18,570 & 0.098 & 0.107*** (0.017) & 1.10 & -0.019 (0.023) & 107 & 49 \\
subclass (4) & 492 & 27,440 & 0.079 & 0.061*** (0.015) & 0.78 & -0.014 (0.028) & 127 & 56 \\
\bottomrule
\end{tabular}
\end{table}

\footnotesize Dependent variable: new patent family in the cell. Regressor: network present (0/1). SEs clustered by firm. $^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$.


### Long differences and selection into exposure

Ten annual observations of a near-monotone cumulative stock against a persistent outcome is
the Bertrand–Duflo–Mullainathan setup: serial correlation in both $x$ and $y$ means
clustered standard errors can overstate precision. Collapsing to two periods is the standard
fix — one pre/post contrast in place of ten correlated annual ones.

Periods are 2015–2019 and 2020–2024, on the 4-character risk set (the notebook 5 baseline).
For each firm-subclass pair, $y_p = 1$ if the firm filed in that subclass at any point in
period $p$, and $x_p$ is network presence at the start of the period. Differencing removes
the pair effect, so the specification is

$$\Delta y_{is} = \beta\,\Delta x_{is} + \alpha_i + \gamma_s + \varepsilon_{is}$$

with firm and subclass FE absorbing what firm × year and subclass × year absorbed in levels.

Note that `net_stock` is cumulative and therefore never falls, so $\Delta x \in \{0, 1\}$:
either the network entered between periods, or nothing changed.

This does not add power — the pairs whose stock changes between periods are the same ones
whose stock changes year to year. What it tests is whether the annual specification was
overstating precision, and, as it turns out, something else entirely.

In [10]:
pre4 = pre.copy()
pre4["sub"] = pre4["ipc_code__main"].str[:4]
keep4 = set(map(tuple, pre4.loc[(pre4["sub"] != "") & pre4[PFIRM].isin(set(panel[FIRM])),
                                [PFIRM, "sub"]].drop_duplicates().values))
rs4 = panel[[(f, s) in keep4 for f, s in zip(panel[FIRM], panel[SUB])]].copy()
rs4["pair"] = rs4[FIRM] + "|" + rs4[SUB]
rs4["period"] = np.where(rs4[YEAR] <= 2019, 1, 2)

ld = (rs4.groupby([FIRM, SUB, "pair", "period"], as_index=False)
         .agg(y=(Y, "max"), x0=("net_any", "first")))

w = ld.pivot(index=[FIRM, SUB, "pair"], columns="period", values=["y", "x0"]).reset_index()
w.columns = [a if not b else f"{a}{b}" for a, b in w.columns]
w["dy"], w["dx"] = w["y2"] - w["y1"], w["x02"] - w["x01"]
w["firm_id"] = w[FIRM].astype("category").cat.codes

# first year the network is present in the pair — needed for the timing split below
w = w.join(rs4[rs4["net_any"] == 1].groupby("pair")[YEAR].min().rename("t0"), on="pair")

print(f"{len(w):,} pairs | {w['firm_id'].nunique():,} firms | dx=1 in {(w.dx == 1).sum():,}")

m_ld = pf.feols(f"dy ~ dx | {FIRM} + {SUB}", data=w, vcov={"CRV1": "firm_id"})
m_ld.tidy()

2,744 pairs | 516 firms | dx=1 in 77


c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 312 singleton fixed effect(s) dropped from the model.
  warnings.warn(


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
dx,-0.091246,0.053897,-1.692972,0.091425,-0.197279,0.014787


The coefficient is −0.091 (0.054), marginally significant at 10% and negative — the opposite
sign to every cross-sectional estimate in notebooks 5 and 6.

Do not read it as a negative spillover. Because $\Delta x$ can only be 0 or 1, the
comparison group pools two opposite populations: pairs exposed in *both* periods and pairs
exposed in *neither*. Those have very different baseline patenting, and with a binary outcome
a group starting above its own mean can mostly only fall. The next cell separates them.

In [11]:
w.groupby(["x01", "x02"]).agg(n=("dy", "size"), y1=("y1", "mean"),
                              y2=("y2", "mean"), dy=("dy", "mean"))

n        y1        y2        dy
x01 x02                                    
0   0    2059  0.180670  0.173385 -0.007285
    1      77  0.337662  0.259740 -0.077922
1   1     608  0.289474  0.250000 -0.039474

The three groups are not comparable, and the entrants are the striking row.

| | n | $y_1$ | $y_2$ | $\Delta y$ |
|---|---|---|---|---|
| Never exposed (0 -> 0) | 2,059 | 0.181 | 0.173 | −0.007 |
| Entrants (0 -> 1) | 77 | 0.338 | 0.260 | −0.078 |
| Always exposed (1 -> 1) | 608 | 0.289 | 0.250 | −0.039 |

Entrants and the never-exposed both start with no network presence in that technology, yet
the entrants were already patenting at 0.338 against 0.181 — roughly double, *before* the
network arrives. Exposure is not assigned to a random subset of a firm's technologies; it
lands where the affiliate is already most active.

That also disposes of the negative coefficient. The pooled comparison group mixes the
never-exposed, who sit flat at the floor, with the always-exposed, who start high and drift
down. Entrants start highest of all and fall most — mean reversion from a selected base,
not an effect of exposure.

Two things follow, and the next cells separate them: whether the negative sign survives a
clean comparison group, and whether the baseline gap is large enough to matter on its own.

In [12]:
w0 = w[w.x01 == 0].copy()

m_ld2 = pf.feols(f"dy ~ dx | {FIRM} + {SUB}", data=w0, vcov={"CRV1": "firm_id"})
m_pre = pf.feols(f"y1 ~ dx | {FIRM} + {SUB}", data=w0, vcov={"CRV1": "firm_id"})

print(w0.groupby("dx")["y1"].agg(["mean", "size"]), "\n")
print(m_ld2.tidy(), "\n")
print(m_pre.tidy())

        mean  size
dx                
0   0.180670  2059
1   0.337662    77 

             Estimate  Std. Error   t value  Pr(>|t|)      2.5%    97.5%
Coefficient                                                             
dx          -0.119969    0.064308 -1.865555  0.063247 -0.246608  0.00667 

             Estimate  Std. Error  t value  Pr(>|t|)      2.5%     97.5%
Coefficient                                                             
dx           0.197107    0.066933  2.94482  0.003529  0.065297  0.328917


c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 343 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 343 singleton fixed effect(s) dropped from the model.
  warnings.warn(


Restricting to pairs unexposed in 2015 — entrants against the never-exposed, a clean
comparison — the difference estimate is −0.120 (0.064), slightly larger in magnitude than
the pooled version. So the negative sign is not an artifact of the mixed comparison group.
It is mean reversion: entrants start at 0.338, a full standard deviation of the pair
distribution above the never-exposed 0.181, and a binary outcome starting that high can
mostly only fall.

The placebo makes that explicit. Regressing *pre-period* patenting on future entry, within
firm and subclass, gives **+0.197 (0.067), p = 0.004**. Network entry is predicted by what
the affiliate was already doing in that technology before the network arrived — and the
selection is sharper within a firm than the raw gap of 0.157 suggests, so this is not merely
"active firms belong to active groups."

For scale: the cross-sectional exposure effect in notebook 5 was 0.061. The selection into
exposure is three times that. The samples and windows differ, so it is not a like-for-like
comparison, but it establishes that selection of the relevant size is present — more than
enough to generate the cross-sectional association without any knowledge flowing.

One caveat before this is read as anticipation. `x02` is network presence in 2020, so entry
occurred somewhere in 2015–2019 — inside the window $y_1$ measures. Part of the 0.197 is
therefore contemporaneous rather than forward-looking. The next cell restricts entry to
2020 and after, so the placebo is strictly anticipatory.

In [13]:
# Entry strictly after the y1 window, so the placebo is anticipatory
clean = w0[w0["t0"].isna() | (w0["t0"] >= 2020)].copy()
clean["late"] = (clean["t0"] >= 2020).fillna(False).astype(int)

print(clean.groupby("late")["y1"].agg(["mean", "size"]), "\n")
m_pre2 = pf.feols(f"y1 ~ late | {FIRM} + {SUB}", data=clean, vcov={"CRV1": "firm_id"})
print(m_pre2.tidy())

          mean  size
late                
0     0.177203  2009
1     0.347826    69 

             Estimate  Std. Error   t value  Pr(>|t|)      2.5%     97.5%
Coefficient                                                              
late         0.076257    0.078023  0.977368  0.329339 -0.077415  0.229929


c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 338 singleton fixed effect(s) dropped from the model.
  warnings.warn(


Restricting entry to 2020 and after, so the pre-period window is strictly before exposure,
the raw gap holds: 0.348 against 0.177 on 69 late entrants. Conditional on firm and subclass
FE, the estimate is **+0.076 (0.078)** — same sign, no precision, p = 0.33.

The drop from 0.197 to 0.076 is not a contradiction but a decomposition. The raw gap is
largely firm-level: firms that patent heavily belong to groups whose networks patent
heavily, and the FE absorb exactly that. What remains is the narrower question — within a
given firm, across its own technologies, does the network enter where that firm is already
more active? The point estimate says probably yes, by about 8pp. Sixty-nine pairs cannot
say so with confidence.

Where that leaves the section:

- **Established.** Exposure is strongly selected. Pairs the network enters were already
  patenting at roughly twice the rate of pairs it never enters, before entry.
- **Established.** The negative long-difference coefficient is mean reversion from that
  selected base, not evidence of a negative effect.
- **Unresolved.** How much of the selection is anticipatory within a firm, as opposed to a
  firm-level correlation between a group's and an affiliate's overall patenting intensity.
  The strictly anticipatory estimate is positive but imprecise.

The unresolved item fails for the same reason everything on the time dimension fails in this
data: roughly 70–130 firm-technology pairs see the network enter inside the window. That
constraint has now appeared in four separate forms — the pair-FE arm in notebook 5, the
aggregation ladder, the long difference, and this placebo — and it is a property of the
sample, not of any specification choice.

### Inference check

Same three variance estimators as notebooks 4 and 5, so the three notebooks are comparable:
firm-level CRV1, heteroskedasticity-robust (HC1, no clustering), and parent-group
clustering. Run on the cross-sectional specification, since that is where the significant
coefficients are.

In [14]:
inf6 = []
for v in [X, "log_net", "net_any"]:
    inf6.append(pf.feols(f"{Y} ~ {v} | {fe1}", data=rsp3, vcov={"CRV1": "firm_id"}))
    inf6.append(pf.feols(f"{Y} ~ {v} | {fe1}", data=rsp3, vcov="hetero"))
    inf6.append(pf.feols(f"{Y} ~ {v} | {fe1}", data=rsp3, vcov={"CRV1": PARENT}))

it6 = pf.etable(inf6, type="df").drop("fe", level=0)

it6.loc[("spec", "Standard errors"), :]  = ["Firm", "Robust", "Parent group"] * 3
it6.loc[("fe", "Firm x year FE"), :]     = ["Yes"] * 9
it6.loc[("fe", "Class x year FE"), :]    = ["Yes"] * 9
it6.loc[("fe", "Firm-class FE"), :]      = ["No"] * 9

it6 = it6.reindex(["coef", "spec", "fe", "stats"], level=0)
it6 = it6.replace({r"\n": " "}, regex=True)
it6

c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2180 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\

patented                      \
                                     (1)                 (2)   
coef  net_stock        0.000**   (0.000)  0.000***   (0.000)   
      log_net                                                  
      net_any                                                  
spec  Standard errors               Firm              Robust   
fe    Firm x year FE                 Yes                 Yes   
      Class x year FE                Yes                 Yes   
      Firm-class FE                   No                  No   
stats Observations                16,390              16,390   
      R²                            0.31                0.31   

                                                              \
                                     (3)                 (4)   
coef  net_stock        0.000**   (0.000)                       
      log_net                             0.057***   (0.009)   
      net_any                                                  
spec  Standard errors       Parent group                Firm   
fe    Firm x year FE                 Yes                 Yes   
      Class x year FE                Yes                 Yes   
      Firm-class FE                   No                  No   
stats Observations                16,390              16,390   
      R²                            0.31               0.329   

                                                               \
                                      (5)                 (6)   
coef  net_stock                                                 
      log_net          0.057***   (0.003)  0.057***   (0.009)   
      net_any                                                   
spec  Standard errors              Robust        Parent group   
fe    Firm x year FE                  Yes                 Yes   
      Class x year FE                 Yes                 Yes   
      Firm-class FE                    No                  No   
stats Observations                 16,390              16,390   
      R²                            0.329               0.329   

                                                               \
                                      (7)                 (8)   
coef  net_stock                                                 
      log_net                                                   
      net_any          0.107***   (0.017)  0.107***   (0.007)   
spec  Standard errors                Firm              Robust   
fe    Firm x year FE                  Yes                 Yes   
      Class x year FE                 Yes                 Yes   
      Firm-class FE                    No                  No   
stats Observations                 16,390              16,390   
      R²                            0.322               0.322   

                                           
                                      (9)  
coef  net_stock                            
      log_net                              
      net_any          0.107***   (0.018)  
spec  Standard errors        Parent group  
fe    Firm x year FE                  Yes  
      Class x year FE                 Yes  
      Firm-class FE                    No  
stats Observations                 16,390  
      R²                            0.322

In [15]:
iout6 = it6.copy()
iout6.index = iout6.index.droplevel(0)
iout6 = iout6.rename(index={X: "Network stock", "log_net": "log(1 + stock)",
                            "net_any": "Network present", "R²": "$R^2$"})

print(iout6.to_latex(escape=False, column_format="l" + "c" * 9,
                     caption="IPC 3-character class under alternative standard errors",
                     label="tab:inf_nb6"))

\begin{table}
\caption{IPC 3-character class under alternative standard errors}
\label{tab:inf_nb6}
\begin{tabular}{lccccccccc}
\toprule
 & \multicolumn{9}{r}{patented} \\
 & (1) & (2) & (3) & (4) & (5) & (6) & (7) & (8) & (9) \\
\midrule
Network stock & 0.000**   (0.000) & 0.000***   (0.000) & 0.000**   (0.000) &  &  &  &  &  &  \\
log(1 + stock) &  &  &  & 0.057***   (0.009) & 0.057***   (0.003) & 0.057***   (0.009) &  &  &  \\
Network present &  &  &  &  &  &  & 0.107***   (0.017) & 0.107***   (0.007) & 0.107***   (0.018) \\
Standard errors & Firm & Robust & Parent group & Firm & Robust & Parent group & Firm & Robust & Parent group \\
Firm x year FE & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes \\
Class x year FE & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes \\
Firm-class FE & No & No & No & No & No & No & No & No & No \\
Observations & 16,390 & 16,390 & 16,390 & 16,390 & 16,390 & 16,390 & 16,390 & 16,390 & 16,390 \\
$R^2$ & 0.31 & 0.31 & 0.31 & 0.329 & 0.329 & 0.329

Inference is stable. `net_any` is 0.107 and significant at 0.1% under all three estimators,
with SEs of 0.017 (firm), 0.007 (HC1) and 0.018 (parent group); `log_net` is 0.057
throughout. Parent-group clustering widens the interval only marginally over firm level, the
same pattern as notebooks 4 and 5 — residual dependence within corporate groups is small
once the fixed effects are in.

One difference from the earlier notebooks: the levels form is significant under all three
estimators here, rather than flipping between them as it did at subclass level. At this
aggregation the coefficient is still unreadable at three decimals, but it is at least
behaving consistently.

## Summary

Notebook 6 set out to test whether the time-dimension null in notebook 5 was a feature of
the data or of how technology was defined. It is the data. Along the way the long-difference
exercise turned up the most direct evidence in the project.

**The aggregation ladder.** Running the cross-sectional specification at section (8 units),
class (119) and subclass (492) gives coefficients of 0.160, 0.107 and 0.061 — rising
monotonically with aggregation. Localized knowledge flow predicts the opposite: widening the
category should dilute a real spillover by mixing proximity with mere co-membership of a
broad field. Sorting predicts exactly this, since firms and their siblings share a broad
field more reliably than a narrow one. The relative effect (0.78, 1.10, 1.10) rises then
plateaus, so the direction is consistent but the pattern is a step rather than a gradient. The cross-sectional coefficients hold at 0.107 and 0.057 under firm, HC1 and parent-group
standard errors alike, so none of this turns on the clustering choice.

**The within-pair arm is flat everywhere.** −0.014, −0.019, −0.011 across the three levels,
on 127, 107 and 105 moving pairs. Aggregation moves the cross-sectional estimate
substantially and leaves the time margin untouched, because merging technologies repackages
existing changes in network stock rather than creating new ones.

**Selection into exposure.** Among pairs unexposed in 2015, those the network later enters
were already patenting at 0.338 against 0.181 for those it never enters — double the rate,
before the network arrives. Regressing pre-period patenting on future entry within firm and
subclass gives +0.197 (0.067, p = 0.004); restricting to strictly post-window entry gives
+0.076 (0.078) on 69 pairs, same sign without precision. Most of the raw gap is firm-level
sorting; the within-firm anticipatory component is positive but unresolved.

**The negative long-difference coefficient is mean reversion.** −0.120 (0.064) on the clean
comparison, driven by entrants starting high and falling, not by exposure reducing patenting.

**What this adds to the project.** Three pieces of evidence now point the same way, and they
are independent of each other: intra-group citation is rare (nb 2); the cross-sectional
association vanishes under firm-subclass FE (nb 5); the association strengthens with
technological aggregation, and exposure lands where the affiliate is already active (here).
The last is the strongest, because it is a direct demonstration of selection rather than an
inference from a coefficient collapsing.

**What stays out of reach.** Roughly 70–130 firm-technology pairs see the network enter
during 2015–2024. That single constraint defeated the within-pair arm in notebook 5, both
variants here, and the anticipatory placebo. Notebook 7's instrument would be estimated on
the same pairs and cannot escape it. Opening the time margin needs a longer window or a
wider network definition — the parent-portfolio route ruled out on scope grounds — and that
is a design decision, not something another specification can solve.